In [4]:
import numpy as np
import torch
import torch.nn as nn

# Generate sample stock-like data
np.random.seed(42)

data = np.sin(
    np.linspace(0, 50, 500)
) + np.random.normal(
    0, 0.05, 500
)

# Convert to PyTorch tensor
data = torch.tensor(
    data,
    dtype=torch.float32
)

# Create sequences
sequence_length = 20

X = []
Y = []

for i in range(
    len(data) - sequence_length
):
    X.append(
        data[i:i + sequence_length]
    )
    Y.append(
        data[i + sequence_length]
    )

X = torch.stack(X)
Y = torch.stack(Y)

# Add feature dimension
X = X.unsqueeze(2)

# LSTM Model
class StockLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=50,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Linear(50, 1)

    def forward(self, x):

        output, _ = self.lstm(x)

        last_output = output[:, -1, :]

        prediction = self.fc(
            last_output
        )

        return prediction


model = StockLSTM()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Training
epochs = 20

for epoch in range(epochs):

    optimizer.zero_grad()

    prediction = model(X)

    loss = criterion(
        prediction.squeeze(),
        Y
    )

    # Backpropagation Through Time
    loss.backward()

    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(
            "Epoch:",
            epoch + 1,
            "Loss:",
            round(loss.item(), 6)
        )

# Prediction
test_input = X[-1].unsqueeze(0)

predicted = model(test_input)

print("\nPredicted next value:",
      round(predicted.item(), 4))

Epoch: 5 Loss: 0.477893
Epoch: 10 Loss: 0.442202
Epoch: 15 Loss: 0.385133
Epoch: 20 Loss: 0.286651

Predicted next value: -0.2892
